In [ ]:
import rbfopt
import numpy as np
import csv
import time

In [40]:
def load_grid(filename="reactor_original_data.csv"):
    """
    Load grid data from CSV.
    Expected columns: Temperature_K, Time_s, Route, C_final
    Returns a dict mapping (Temperature_K, Time_s, Route) -> C_final
    """
    grid_data = {}
    with open(filename, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                x_val = float(row["Temperature_K"])     # keep as float
                y_val = float(row["Time_s"])            # keep as float
                z_val = int(float(row["Route"]))        # "1" or "1.0" -> 1
                obj_val = float(row["C_final"])
                grid_data[(x_val, y_val, z_val)] = obj_val
            except (KeyError, ValueError, TypeError):
                continue
    return grid_data

grid_data = load_grid()

xs = [pt[0] for pt in grid_data.keys()]
ys = [pt[1] for pt in grid_data.keys()]
zs = [pt[2] for pt in grid_data.keys()]

min_x, max_x = min(xs), max(xs)
min_y, max_y = min(ys), max(ys)
min_z, max_z = min(zs), max(zs)

print("Grid bounds:")
print("Temperature_K in [{}, {}]".format(min_x, max_x))
print("Time_s in [{}, {}]".format(min_y, max_y))
print("Route in [{}, {}]".format(min_z, max_z))

Grid bounds:
Temperature_K in [100.0, 600.0]
Time_s in [1.0, 120.0]
Route in [1, 2]


In [41]:
# Load the data from grid.csv
grid_data = load_grid()

PENALTY = 1e5  # adjust this penalty as appropriate
# PENALTY = np.inf # use infinity as penalty for infeasible points
# PENALTY = np.nan  # use NaN as penalty for infeasible points
# PENALTY = None  # use None as penalty for infeasible points

# Extract allowed grid values for snapping
T_vals = sorted({pt[0] for pt in grid_data.keys()})
t_vals = sorted({pt[1] for pt in grid_data.keys()})

def _nearest(val, arr):
    return min(arr, key=lambda a: abs(a - val))


def obj_funct(x):
    """
    Snap continuous candidate values to nearest grid points
    and return objective.
    """

    # Raw candidate values from RBFOpt
    x_raw = float(x[0])
    y_raw = float(x[1])
    z_raw = float(x[2]) if len(x) > 2 else 1.0

    # Snap Temperature and Time to nearest available values in CSV
    x_val = _nearest(x_raw, T_vals)
    y_val = _nearest(y_raw, t_vals)

    # Route must be 1 or 2
    z_val = int(round(z_raw))
    z_val = max(1, min(2, z_val))

    return grid_data.get((x_val, y_val, z_val), PENALTY)


In [42]:
test_point = [100.0,7.2631578947368425,1]
value = obj_funct(test_point)
print(f"For point {test_point} the objective value is {value}")

# 100.0,7.2631578947368425,1,2.3771385601402317e-32

For point [100.0, 7.2631578947368425, 1] the objective value is 2.3771385601402317e-32


In [43]:
start_time = time.time()

black_box = rbfopt.RbfoptUserBlackBox(
    3,                   # number of variables
    np.array([100, 1, 1]),    # lower bounds for x, y, z
    np.array([600, 120, 2]),    # upper bounds for x, y, z
    np.array(["R", "R", "I"]), # variable types: "I" for integer, "R" for real
    obj_funct
)

# Configure optimization settings.
settings = rbfopt.RbfoptSettings(
    max_evaluations=100,  # maximum number of evaluations
    minlp_solver_path='/home/albertlee/ampl.linux-intel64/bonmin',
    nlp_solver_path='/home/albertlee/ampl.linux-intel64/ipopt',
    algorithm='MSRSM',   # optimization algorithm variant
    rbf='auto',           # allow automatic RBF selection
    global_search_method = 'solver',  # options = ['sampling', 'genetic', 'solver']
)

# Create the optimization algorithm instance.
alg = rbfopt.RbfoptAlgorithm(settings, black_box)

# Run the optimization.
best_val, best_point, itercount, evalcount, fast_evalcount = alg.optimize()
end_time = time.time()

print(f"\nTotal optimization time: {end_time - start_time:.2f} seconds")
print("Optimization completed.")
print("Best objective value found:", best_val)

best_T = _nearest(float(best_point[0]), T_vals)
best_t = _nearest(float(best_point[1]), t_vals)
best_route = int(round(float(best_point[2])))
best_route = max(1, min(2, best_route))

print("At lattice point (actual values):", (best_T, best_t, best_route))
print("Objective at snapped point:", grid_data.get((best_T, best_t, best_route)))
print("Iterations:", itercount, "Function evaluations:", evalcount)

  Iter  Cycle  Action             Objective value      Time      Gap
  ----  -----  ------             ---------------      ----      ---
     0      0  Initialization            0.000000      0.01   100.00  
     0      0  Initialization           -0.000000      0.01   100.00 *
     0      0  Initialization            0.902043      0.01   100.00  
     0      0  GlobalStep                0.484632      0.29   100.00  
     1      0  GlobalStep                0.000000      0.45   100.00  
     2      0  GlobalStep                0.000000      0.83   100.00  
     3      0  GlobalStep                0.000000      1.01   100.00  
     4      0  GlobalStep                1.000000      1.32   100.00  
     5      0  AdjLocalStep              0.000000      2.03   100.00  
     6      1  GlobalStep                0.000000      2.33   100.00  
     7      1  GlobalStep                0.016611      2.58   100.00  
     8      1  GlobalStep                0.025572      2.82   100.00  
     9    

In [49]:
import math
import time
from typing import Dict, Iterable, List, Optional, Tuple

import pandas as pd

Point3 = Tuple[int, int, int]  # (i_temp, j_time, k_route)


# =============================
# CSV -> (i,j,k) -> value mapping
# plus maps back to original values
# =============================
def load_grid_3d_from_csv(
    csv_path: str,
    value_col: str = "C_final",
    temp_col: str = "Temperature_K",
    time_col: str = "Time_s",
    route_col: str = "Route",
    start_index: int = 1,   # <-- makes the lowest index = 1
) -> Tuple[Dict[Point3, float], dict, dict, dict]:
    df = pd.read_csv(csv_path)

    needed = {temp_col, time_col, route_col, value_col}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"CSV missing required columns: {sorted(missing)}")

    # Unique sorted values -> integer indices (1..n)
    temps = sorted(df[temp_col].unique().tolist())
    times = sorted(df[time_col].unique().tolist())
    routes = sorted(df[route_col].unique().tolist())

    temp_to_i = {v: idx for idx, v in enumerate(temps, start=start_index)}
    time_to_j = {v: idx for idx, v in enumerate(times, start=start_index)}
    route_to_k = {v: idx for idx, v in enumerate(routes, start=start_index)}

    i_to_temp = {i: v for v, i in temp_to_i.items()}
    j_to_time = {j: v for v, j in time_to_j.items()}
    k_to_route = {k: v for v, k in route_to_k.items()}

    grid: Dict[Point3, float] = {}
    for _, r in df.iterrows():
        i = temp_to_i[r[temp_col]]
        j = time_to_j[r[time_col]]
        k = route_to_k[r[route_col]]
        grid[(i, j, k)] = float(r[value_col])

    if not grid:
        raise ValueError("CSV parsed but produced an empty 3D grid.")

    maps = {
        "temp_to_i": temp_to_i, "time_to_j": time_to_j, "route_to_k": route_to_k,
        "i_to_temp": i_to_temp, "j_to_time": j_to_time, "k_to_route": k_to_route,
        "I": range(start_index, start_index + len(temps)),
        "J": range(start_index, start_index + len(times)),
        "K": range(start_index, start_index + len(routes)),
    }
    return grid, maps


# =============================
# Neighbor utilities (3D, Linf neighborhood)
# =============================
def get_neighbors_3d(
    I: Iterable[int],
    J: Iterable[int],
    K: Iterable[int],
    x: int,
    y: int,
    z: int,
    forbid_x: Optional[int] = None,  # optionally forbid a temp index
    forbid_y: Optional[int] = None,  # optionally forbid a time index
    forbid_z: Optional[int] = None,  # optionally forbid a route index
    include_self: bool = False,
) -> List[Point3]:
    directions = [(dx, dy, dz) for dx in (-1, 0, 1)
                             for dy in (-1, 0, 1)
                             for dz in (-1, 0, 1)]
    if not include_self:
        directions = [d for d in directions if d != (0, 0, 0)]

    I_set, J_set, K_set = set(I), set(J), set(K)

    nbrs: List[Point3] = []
    for dx, dy, dz in directions:
        nx, ny, nz = x + dx, y + dy, z + dz
        if nx in I_set and ny in J_set and nz in K_set:
            if forbid_x is not None and nx == forbid_x:
                continue
            if forbid_y is not None and ny == forbid_y:
                continue
            if forbid_z is not None and nz == forbid_z:
                continue
            nbrs.append((nx, ny, nz))
    return nbrs


def _step_length_3d(p: Point3, q: Point3) -> float:
    return math.sqrt((q[0]-p[0])**2 + (q[1]-p[1])**2 + (q[2]-p[2])**2)


def pick_best_neighbor_3d(
    current: Point3,
    candidates: List[Point3],
    f: Dict[Point3, float],
    prefer_farther_on_tie: bool = True,
) -> Point3:
    best = candidates[0]
    best_val = f[best]

    for q in candidates[1:]:
        v = f[q]
        if v < best_val:
            best, best_val = q, v
        elif v == best_val:
            if prefer_farther_on_tie:
                dq = _step_length_3d(current, q)
                db = _step_length_3d(current, best)
                if dq > db:
                    best, best_val = q, v
                elif dq == db and q < best:
                    best, best_val = q, v
            else:
                if q < best:
                    best, best_val = q, v
    return best


# =============================
# Strict LDSDA line search (3D)
# =============================
def line_search_strict_3d(
    start: Point3,
    direction: Point3,  # (dx,dy,dz) in {-1,0,1}^3, not (0,0,0)
    f: Dict[Point3, float],
    I_set: set,
    J_set: set,
    K_set: set,
    forbid_x: Optional[int] = None,
    forbid_y: Optional[int] = None,
    forbid_z: Optional[int] = None,
    eps: float = 0.0,
    max_steps: int = 10_000,
) -> Tuple[Point3, List[Point3]]:
    dx, dy, dz = direction
    if (dx, dy, dz) == (0, 0, 0):
        return start, [start]

    curr = start
    curr_val = f[curr]
    segment = [curr]

    for _ in range(max_steps):
        nxt = (curr[0] + dx, curr[1] + dy, curr[2] + dz)

        if nxt[0] not in I_set or nxt[1] not in J_set or nxt[2] not in K_set:
            break
        if forbid_x is not None and nxt[0] == forbid_x:
            break
        if forbid_y is not None and nxt[1] == forbid_y:
            break
        if forbid_z is not None and nxt[2] == forbid_z:
            break
        if nxt not in f:
            break

        nxt_val = f[nxt]
        if nxt_val < curr_val - eps:
            curr = nxt
            curr_val = nxt_val
            segment.append(curr)
        else:
            break

    return curr, segment


# =============================
# Main LD-SDA loop (3D)
# =============================
def ld_sda_3d(
    I: Iterable[int],
    J: Iterable[int],
    K: Iterable[int],
    initial_point: Point3,
    grid_values: Dict[Point3, float],
    forbid_x: Optional[int] = None,
    forbid_y: Optional[int] = None,
    forbid_z: Optional[int] = None,
    eps: float = 0.0,
    prefer_farther_on_tie: bool = True,
    max_iters: int = 10_000,
    max_line_steps: int = 10_000,
    verbose: bool = True,
    trace: bool = True,
) -> Dict[str, object]:
    I_set, J_set, K_set = set(I), set(J), set(K)

    if initial_point not in grid_values:
        raise ValueError(f"Initial point {initial_point} not found in the grid values.")
    if initial_point[0] not in I_set or initial_point[1] not in J_set or initial_point[2] not in K_set:
        raise ValueError(f"Initial point {initial_point} not inside I×J×K domain.")

    current = initial_point
    path: List[Point3] = [current]
    line_segments: List[List[Point3]] = []
    neighbor_trace: List[dict] = []

    if verbose:
        print(f"[init] point={current}, f={grid_values[current]:.6e}")

    for it in range(1, max_iters + 1):
        curr_val = grid_values[current]

        nbrs = get_neighbors_3d(
            I_set, J_set, K_set,
            current[0], current[1], current[2],
            forbid_x=forbid_x, forbid_y=forbid_y, forbid_z=forbid_z,
            include_self=False,
        )
        nbrs = [p for p in nbrs if p in grid_values]
        if not nbrs:
            if verbose:
                print(f"[stop] iter={it}: no neighbors in grid.")
            break

        if verbose:
            print(f"         neighbors ({len(nbrs)}):")
            for p in nbrs:
                print(f"           {p} -> f={grid_values[p]:.6e}")

        best_nbr = pick_best_neighbor_3d(current, nbrs, grid_values, prefer_farther_on_tie)
        best_val = grid_values[best_nbr]

        if best_val >= curr_val - eps:
            if verbose:
                print(f"[stop] iter={it}: surrounded (best={best_val:.6e} >= curr={curr_val:.6e}).")
            break

        direction = (best_nbr[0] - current[0], best_nbr[1] - current[1], best_nbr[2] - current[2])

        if verbose:
            print(
                f"[iter {it}] current={current} f={curr_val:.6e}\n"
                f"         --> chosen neighbor={best_nbr} f={best_val:.6e} dir={direction}"
            )

        if trace:
            neighbor_trace.append(
                {
                    "iter": it,
                    "current": current,
                    "current_value": curr_val,
                    "neighbors": [(p, grid_values[p]) for p in nbrs],
                    "chosen_neighbor": best_nbr,
                    "chosen_value": best_val,
                    "direction": direction,
                }
            )

        new_point, segment = line_search_strict_3d(
            start=current,
            direction=direction,
            f=grid_values,
            I_set=I_set,
            J_set=J_set,
            K_set=K_set,
            forbid_x=forbid_x,
            forbid_y=forbid_y,
            forbid_z=forbid_z,
            eps=eps,
            max_steps=max_line_steps,
        )

        line_segments.append(segment)
        if len(segment) > 1:
            path.extend(segment[1:])

        if verbose:
            print(f"         line_search: len={len(segment)} end={new_point} f_end={grid_values[new_point]:.6e}")
            print("         segment:")
            for p in segment:
                print(f"           {p} -> f={grid_values[p]:.6e}")

        if new_point == current:
            if verbose:
                print("         (fallback) line search didn't move; stepping to best_nbr.")
            current = best_nbr
            path.append(current)
        else:
            current = new_point

    return {
        "best_point": current,
        "best_value": grid_values[current],
        "path": path,
        "line_segments": line_segments,
        "neighbor_trace": neighbor_trace,
        "iterations": len(line_segments),
    }


# =============================
# Example run on your reactor CSV
# =============================
csv_path = "reactor_original_data.csv"

grid3d, maps = load_grid_3d_from_csv(
    csv_path,
    value_col="C_final",
    temp_col="Temperature_K",
    time_col="Time_s",
    route_col="Route",
    start_index=1,   # <-- lowest index = 1
)

I = maps["I"]
J = maps["J"]
K = maps["K"]

# Pick an initial lattice point (1,1,1) = (lowest temp, lowest time, lowest route)
initial_point = (15, 8, 1)

t0 = time.time()
res = ld_sda_3d(
    I=I, J=J, K=K,
    initial_point=initial_point,
    grid_values=grid3d,
    forbid_x=None, forbid_y=None, forbid_z=None,
    eps=0.0,
    prefer_farther_on_tie=True,
    max_iters=10_000,
    max_line_steps=10_000,
    verbose=True,
    trace=True,
)
t1 = time.time()

best_idx = res["best_point"]
best_T = maps["i_to_temp"][best_idx[0]]
best_time = maps["j_to_time"][best_idx[1]]
best_route = maps["k_to_route"][best_idx[2]]

print(f"\nTotal time: {t1 - t0:.2f} seconds")
print("\n=== RESULT (3D) ===")
print("Best lattice point (i,j,k):", best_idx)
print("Best value:", res["best_value"])
print("Best original (Temperature_K, Time_s, Route):", (best_T, best_time, best_route))
print("Iterations:", res["iterations"])
print("Path length:", len(res["path"]))
print("Path (first 30):", res["path"][:30])


[init] point=(15, 8, 1), f=9.867268e-01
         neighbors (17):
           (14, 7, 1) -> f=8.378726e-01
           (14, 7, 2) -> f=1.160227e-01
           (14, 8, 1) -> f=8.846447e-01
           (14, 8, 2) -> f=8.159508e-02
           (14, 9, 1) -> f=9.179764e-01
           (14, 9, 2) -> f=5.732992e-02
           (15, 7, 1) -> f=9.746114e-01
           (15, 7, 2) -> f=1.852494e-02
           (15, 8, 2) -> f=9.652941e-03
           (15, 9, 1) -> f=9.930603e-01
           (15, 9, 2) -> f=5.029859e-03
           (16, 7, 1) -> f=9.988036e-01
           (16, 7, 2) -> f=1.006678e-03
           (16, 8, 1) -> f=9.996218e-01
           (16, 8, 2) -> f=3.266672e-04
           (16, 9, 1) -> f=9.998805e-01
           (16, 9, 2) -> f=1.066934e-04
[iter 1] current=(15, 8, 1) f=9.867268e-01
         --> chosen neighbor=(16, 9, 2) f=1.066934e-04 dir=(1, 1, 1)
         line_search: len=2 end=(16, 9, 2) f_end=1.066934e-04
         segment:
           (15, 8, 1) -> f=9.867268e-01
           (16, 9, 2) -